# TFG · Lección 2 (sistemas múltiples): comprobaciones en Qiskit

Cada bloque comprueba un resultado del Capítulo 2 del cuaderno de computación cuántica. Ejecuta las celdas **en orden**.

**Convención de Qiskit:** el qubit 0 es el de la **derecha**. En un estado de dos qubits $|ab\rangle$ del cuaderno,
$a$ (el sistema $\mathsf{X}$, a la izquierda) es el qubit **1** de Qiskit y $b$ (el sistema $\mathsf{Y}$) es el qubit **0**.

In [1]:
from qiskit import __version__
print(__version__)

2.2.3


In [2]:
import numpy as np
from numpy import sqrt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator, partial_trace

def probs(estado, qargs=None):
    """Probabilidades de Born (de todos los qubits o solo de qargs), redondeadas."""
    return {k: round(float(p), 4) for k, p in estado.probabilities_dict(qargs).items()}

## Experimento 1 · Producto tensorial y orden de las entradas

- `a.tensor(b)` (o `a ^ b`) es $|a\rangle\otimes|b\rangle$: el argumento es el factor de la **derecha**.
- El producto tensorial **no es conmutativo**.
- La entrada de la cadena $a_{n-1}\cdots a_0$ está en la posición del número que representa en binario.

In [3]:
cero, uno = Statevector.from_label("0"), Statevector.from_label("1")

print("|0> ⊗ |1> =", (cero ^ uno).draw("latex_source"))
print("|1> ⊗ |0> =", (uno ^ cero).draw("latex_source"))
print("¿Conmutativo?", (cero ^ uno).equiv(uno ^ cero))

# Posición de |101> en el vector de 3 qubits: debe ser 5 (101 en binario)
v = Statevector.from_label("101")
print("Índice de la entrada no nula de |101>:", int(np.argmax(np.abs(v.data))))

|0> ⊗ |1> =  |01\rangle
|1> ⊗ |0> =  |10\rangle
¿Conmutativo? False
Índice de la entrada no nula de |101>: 5


In [4]:
# |+> ⊗ |-i>, comparado con el cálculo a mano: (1/2)(|00> - i|01> + |10> - i|11>)
mas, menos_i = Statevector.from_label("+"), Statevector.from_label("l")
phi = mas ^ menos_i
a_mano = Statevector(np.array([1, -1j, 1, -1j]) / 2)
print(phi.draw("latex_source"))
print("Coincide con el cálculo a mano:", phi.equiv(a_mano))

\frac{1}{2} |00\rangle- \frac{i}{2} |01\rangle+\frac{1}{2} |10\rangle- \frac{i}{2} |11\rangle
Coincide con el cálculo a mano: True


## Experimento 2 · El producto interno es multiplicativo

$\langle \phi_1\otimes\psi_1|\phi_2\otimes\psi_2\rangle = \langle\phi_1|\phi_2\rangle\langle\psi_1|\psi_2\rangle$.
Es la propiedad que hace que un mapa de características sin entrelazamiento dé un kernel que se factoriza.

In [5]:
rng = np.random.default_rng(0)
def estado_aleatorio():
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    return Statevector(v / np.linalg.norm(v))

f1, f2, p1, p2 = [estado_aleatorio() for _ in range(4)]
izq = np.vdot((f1 ^ p1).data, (f2 ^ p2).data)          # np.vdot conjuga el primer vector
der = np.vdot(f1.data, f2.data) * np.vdot(p1.data, p2.data)
print("Lado izquierdo:", np.round(izq, 6))
print("Lado derecho:  ", np.round(der, 6))
print("Iguales:", np.isclose(izq, der))

Lado izquierdo: (0.40519+0.239176j)
Lado derecho:   (0.40519+0.239176j)
Iguales: True


## Experimento 3 · Detectar el entrelazamiento (criterio del rango)

Colocamos las amplitudes $\alpha_{ab}$ en una matriz $2\times2$ (`reshape`). El estado es producto si y solo si su rango es 1.
Los valores singulares son los coeficientes de Schmidt.

In [6]:
def analizar(nombre, estado):
    A = estado.data.reshape(2, 2)                        # filas: qubit izquierdo (X); columnas: derecho (Y)
    rango = np.linalg.matrix_rank(A, tol=1e-10)
    schmidt = np.linalg.svd(A, compute_uv=False)
    tipo = "PRODUCTO" if rango == 1 else "ENTRELAZADO"
    print(f"{nombre:12s} rango = {rango}   Schmidt = {np.round(schmidt, 4)}   -> {tipo}")

bell  = Statevector(np.array([1, 0, 0, 1]) / sqrt(2))
psi   = Statevector(np.array([1/sqrt(2), -1/sqrt(6), 1j/sqrt(6), 1/sqrt(6)]))
prod  = Statevector(np.array([1, 1j, -1, -1j]) / 2)
t_c   = Statevector(np.array([3/5, 0, 0, 4/5]))

for nombre, e in [("|phi+>", bell), ("psi (3.2.2)", psi), ("producto", prod), ("3/5,4/5", t_c)]:
    analizar(nombre, e)

|phi+>       rango = 2   Schmidt = [0.7071 0.7071]   -> ENTRELAZADO
psi (3.2.2)  rango = 2   Schmidt = [0.9342 0.3568]   -> ENTRELAZADO
producto     rango = 1   Schmidt = [1. 0.]   -> PRODUCTO
3/5,4/5      rango = 2   Schmidt = [0.8 0.6]   -> ENTRELAZADO


## Experimento 4 · Fabricar los estados de Bell con $H$ y CNOT

En el cuaderno, $CX\,(H\otimes\mathbb{I})$ con el qubit **izquierdo** como control. En Qiskit el izquierdo es el qubit 1,
así que el circuito es `h(1)` seguido de `cx(1, 0)`. Debe transformar
$|00\rangle\mapsto|\phi^+\rangle$, $|01\rangle\mapsto|\psi^+\rangle$, $|10\rangle\mapsto|\phi^-\rangle$, $|11\rangle\mapsto|\psi^-\rangle$.

In [7]:
bell_circ = QuantumCircuit(2)
bell_circ.h(1)
bell_circ.cx(1, 0)
display(bell_circ.draw(output="mpl"))
bell_circ.draw(output="mpl").savefig("../apuntes/cuaderno_cuantica/circuito_bell.png", dpi=200, bbox_inches="tight")

esperados = {
    "00": np.array([1, 0, 0, 1]) / sqrt(2),    # phi+
    "01": np.array([0, 1, 1, 0]) / sqrt(2),    # psi+
    "10": np.array([1, 0, 0, -1]) / sqrt(2),   # phi-
    "11": np.array([0, 1, -1, 0]) / sqrt(2),   # psi-
}
for etiqueta, vec in esperados.items():
    resultado = Statevector.from_label(etiqueta).evolve(bell_circ)
    print(f"|{etiqueta}> -> {resultado.draw('latex_source'):45s} correcto: {resultado.equiv(Statevector(vec))}")

<Figure size 287.294x200.667 with 1 Axes>

|00> -> \frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |11\rangle correcto: True
|01> -> \frac{\sqrt{2}}{2} |01\rangle+\frac{\sqrt{2}}{2} |10\rangle correcto: True
|10> -> \frac{\sqrt{2}}{2} |00\rangle- \frac{\sqrt{2}}{2} |11\rangle correcto: True
|11> -> \frac{\sqrt{2}}{2} |01\rangle- \frac{\sqrt{2}}{2} |10\rangle correcto: True


In [8]:
# Aviso sobre el orden: cx(0, 1) (control el qubit DERECHO) no es la matriz CX del cuaderno
cx_cuaderno = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
qc01 = QuantumCircuit(2); qc01.cx(0, 1)
qc10 = QuantumCircuit(2); qc10.cx(1, 0)
print("cx(1,0) == CX del cuaderno:", np.allclose(Operator.from_circuit(qc10).data, cx_cuaderno))
print("cx(0,1) == CX del cuaderno:", np.allclose(Operator.from_circuit(qc01).data, cx_cuaderno))
print(np.real(Operator.from_circuit(qc01).data).astype(int))

cx(1,0) == CX del cuaderno: True
cx(0,1) == CX del cuaderno: False
[[1 0 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [0 1 0 0]]


## Experimento 5 · Mediciones parciales

Estado $|\psi\rangle = \tfrac{1}{\sqrt2}|00\rangle - \tfrac{1}{\sqrt6}|01\rangle + \tfrac{i}{\sqrt6}|10\rangle + \tfrac{1}{\sqrt6}|11\rangle$.
Medir $\mathsf{X}$ (izquierdo) es medir el qubit 1; medir $\mathsf{Y}$ es medir el qubit 0.
En ambos casos, a mano: resultado 0 con probabilidad $2/3$ y 1 con probabilidad $1/3$.

In [9]:
print("Probabilidades al medir X (qubit 1):", probs(psi, [1]))
print("Probabilidades al medir Y (qubit 0):", probs(psi, [0]))

Probabilidades al medir X (qubit 1): {np.str_('0'): 0.6667, np.str_('1'): 0.3333}
Probabilidades al medir Y (qubit 0): {np.str_('0'): 0.6667, np.str_('1'): 0.3333}


In [10]:
# Estados tras la medición, calculados a mano en el cuaderno (orden de entradas: 00, 01, 10, 11)
a_mano = {
    ("X", "0"): np.array([sqrt(3)/2, -1/2, 0, 0]),
    ("X", "1"): np.array([0, 0, 1j/sqrt(2), 1/sqrt(2)]),
    ("Y", "0"): np.array([sqrt(3)/2, 0, 1j/2, 0]),
    ("Y", "1"): np.array([0, -1/sqrt(2), 0, 1/sqrt(2)]),
}
qubit = {"X": 1, "Y": 0}

for (sistema, resultado_buscado), vec in a_mano.items():
    for _ in range(200):                              # repetimos hasta obtener ese resultado
        resultado, estado = psi.measure([qubit[sistema]])
        if resultado == resultado_buscado:
            break
    print(f"Medir {sistema}, resultado {resultado}: coincide con el cálculo a mano -> {estado.equiv(Statevector(vec))}")

Medir X, resultado 0: coincide con el cálculo a mano -> True
Medir X, resultado 1: coincide con el cálculo a mano -> True
Medir Y, resultado 0: coincide con el cálculo a mano -> True
Medir Y, resultado 1: coincide con el cálculo a mano -> True


## Experimento 6 · GHZ frente a W

Medimos el **primer** qubit (el de la izquierda = qubit 2 de Qiskit).
GHZ debe quedar en $|000\rangle$ o $|111\rangle$; W debe quedar en $|0\rangle|\psi^+\rangle$ (prob. 2/3) o en $|100\rangle$ (prob. 1/3).

In [11]:
ghz = Statevector(np.array([1, 0, 0, 0, 0, 0, 0, 1]) / sqrt(2))
w   = Statevector(np.array([0, 1, 1, 0, 1, 0, 0, 0]) / sqrt(3))

print("GHZ, probabilidades del primer qubit:", probs(ghz, [2]))
print("W,   probabilidades del primer qubit:", probs(w, [2]))

for _ in range(5):
    r, e = ghz.measure([2])
    print(f"GHZ -> resultado {r}, estado {e.draw('latex_source')}")
for _ in range(5):
    r, e = w.measure([2])
    print(f"W   -> resultado {r}, estado {e.draw('latex_source')}")

GHZ, probabilidades del primer qubit: {np.str_('0'): 0.5, np.str_('1'): 0.5}
W,   probabilidades del primer qubit: {np.str_('0'): 0.6667, np.str_('1'): 0.3333}
GHZ -> resultado 0, estado  |000\rangle
GHZ -> resultado 1, estado  |111\rangle
GHZ -> resultado 0, estado  |000\rangle
GHZ -> resultado 1, estado  |111\rangle
GHZ -> resultado 0, estado  |000\rangle
W   -> resultado 0, estado \frac{\sqrt{2}}{2} |001\rangle+\frac{\sqrt{2}}{2} |010\rangle
W   -> resultado 0, estado \frac{\sqrt{2}}{2} |001\rangle+\frac{\sqrt{2}}{2} |010\rangle
W   -> resultado 1, estado  |100\rangle
W   -> resultado 1, estado  |100\rangle
W   -> resultado 1, estado  |100\rangle


## Experimento 7 · Estados reducidos: por qué hacen falta las matrices densidad

La fórmula ingenua $\sum_a(\sum_b\alpha_{ab})|a\rangle$ daría $|+\rangle$ para $|\psi^+\rangle$ y $|-\rangle$ para $|\psi^-\rangle$,
que son distinguibles. Pero el estado reducido correcto (traza parcial) de $\mathsf{X}$ es $\tfrac12\mathbb{I}$ en los cuatro estados de Bell.
`partial_trace(estado, [0])` elimina el qubit 0 ($\mathsf{Y}$) y deja la matriz densidad de $\mathsf{X}$.

In [12]:
for etiqueta, nombre in [("00", "phi+"), ("01", "psi+"), ("10", "phi-"), ("11", "psi-")]:
    b = Statevector.from_label(etiqueta).evolve(bell_circ)
    rho_X = partial_trace(b, [0]).data
    print(f"Estado reducido de X en |{nombre}>:")
    print(np.round(rho_X, 4))

Estado reducido de X en |phi+>:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Estado reducido de X en |psi+>:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Estado reducido de X en |phi->:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
Estado reducido de X en |psi->:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


## Experimento 8 · Operaciones de dos qubits

- El producto tensorial de unitarias es unitario.
- $CZ$ es simétrico: da igual qué qubit sea el control.
- $CZ = (\mathbb{I}\otimes H)\,CX\,(\mathbb{I}\otimes H)$.
- El SWAP se puede construir con tres CNOT alternando el control.

In [13]:
H, I, X = Operator.from_label("H"), Operator.from_label("I"), Operator.from_label("X")
U = (H ^ I ^ X).data
print("H⊗I⊗X unitaria:", np.allclose(U.conj().T @ U, np.eye(8)))

cz01 = QuantumCircuit(2); cz01.cz(0, 1)
cz10 = QuantumCircuit(2); cz10.cz(1, 0)
print("CZ simétrico:", Operator.from_circuit(cz01).equiv(Operator.from_circuit(cz10)))

# CZ = (I ⊗ H) CX (I ⊗ H): H sobre el qubit derecho (qubit 0), CX con control el izquierdo (qubit 1)
cz_desde_cx = QuantumCircuit(2)
cz_desde_cx.h(0); cz_desde_cx.cx(1, 0); cz_desde_cx.h(0)
print("CZ = (I⊗H) CX (I⊗H):", Operator.from_circuit(cz_desde_cx).equiv(Operator.from_circuit(cz10)))

swap_3cx = QuantumCircuit(2)
swap_3cx.cx(1, 0); swap_3cx.cx(0, 1); swap_3cx.cx(1, 0)
swap_q = QuantumCircuit(2); swap_q.swap(0, 1)
print("SWAP = tres CNOT:", Operator.from_circuit(swap_3cx).equiv(Operator.from_circuit(swap_q)))
display(swap_3cx.draw(output="mpl"))

H⊗I⊗X unitaria: True
CZ simétrico: True
CZ = (I⊗H) CX (I⊗H): True
SWAP = tres CNOT: True


<Figure size 370.906x200.667 with 1 Axes>